In [ ]:
import os 
import json 
import random 
import hashlib 
from typing import List,Dict
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError
import pandas as pd 

load_dotenv(override=True)

# for validation of job postings
class JobPosting(BaseModel):
    title:str = Field(min_length=3, max_length=120)
    company: str = Field (min_length=2, max_length=120)
    location :str = Field(min_length=2, max_length=80)
    seniority:str
    salary_min:int = Field (ge=20000, le=500000)
    salary_max:int = Field (ge=25000, le=700000)
    skills:List[str]
    description:str = Field(min_length=40, max_length=1200)
 
# for normalizing job postings before validation
    @classmethod 
    def normalize(cls,row:Dict)->Dict:
        row["seniority"] = row.get("seniority","").lower().strip()
        row["skills"] = [skill.strip() for skill in row.get("skills",[]) if str(skill).strip()]
        if row["salary_min"] > row["salary_max"]:
            row["salary_min"], row["salary_max"] = row["salary_max"], row["salary_min"]
        return row

In [ ]:
# for creating the client
def make_client() -> OpenAI:
    api_key = os.getenv("GROQ_API_KEY")
    base_url = "https://api.groq.com/openai/v1"
    return OpenAI(api_key=api_key, base_url=base_url)

SYSTEM_PROMPT = """
You are a synthetic data generator for testing only.
Return a valid JSON object only.
No markdown, no commentary, no code fences, no trailing commas.
Do not include real people, real companies, or private data.
The object must have exactly these keys:
title, company, location, seniority, salary_min, salary_max, skills, description
"""

def build_single_user_prompt(domain:str, seniority_mix:str, index:int) ->str:
    return f"""
Generate 1 synthetic job posting for the domain: {domain}.
Seniority mix: {seniority_mix}.
Record number: {index}.

Rules:
- Return a JSON object only, not an array.
- The object must contain title, company, location, seniority, salary_min, salary_max, skills, description.
- salary_min and salary_max must be integers in USD per year.
- skills must be a JSON array with 4 to 8 items.
- description should be 2 to 5 sentences.
- Keep the data varied across locations, technologies, and company types.
- Do not wrap the response in markdown or extra text.
"""

# for extracting a JSON object from model output, with error handling
def extract_json_object(text: str) -> Dict:
    text = text.strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("Model did not return a JSON object.")
    return json.loads(text[start:end+1])

# for generating synthetic job postings using one-record-at-a-time calls
def generate_batch(client:OpenAI, model:str, n:int, domain:str, seniority_mix, temperature:float =0.2) -> List[Dict]:
    rows: List[Dict] = []
    for index in range(1, n + 1):
        resp = client.chat.completions.create(
            model=model,
            temperature=temperature,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_single_user_prompt(domain, seniority_mix, index)},
            ],
        )
        content = resp.choices[0].message.content or ""
        try:
            row = extract_json_object(content)
        except json.JSONDecodeError:
            continue
        rows.append(row)
    return rows

In [ ]:
# for creating a unique fingerprint for each job posting to identify duplicates
def row_fingerprint(row:Dict) -> str:
    key = f"{row.get('title','')}|{row.get('company','')}|{row.get('location','')}|{','.join(row.get('skills',[]))}"
    return hashlib.md5(key.lower().encode()).hexdigest()

# for validating and deduplicating a list of job postings, returning only valid and unique entries
def validate_row(rows: List[Dict]) -> List[Dict]:
    valid = []
    seen = set()

    for r in rows:
        try:
            r = JobPosting.normalize(r)
            parse = JobPosting(**r).model_dump()
            fp = row_fingerprint(parse)
            if fp not in seen:
                seen.add(fp)
                valid.append(parse)
        except ValidationError:
            continue
    return valid

# for generating a quality report on the validated job postings, including counts, uniqueness, and distributions
def quality_report(rows:List[Dict]) -> Dict:
    if not rows:
        return {"count":0}
    df = pd.DataFrame(rows)
    return {
        "count":len(rows),
        "unique_titles":int(df["title"].nunique()),
        "unique_locations":int(df["location"].nunique()),
        "avg_skills_per_row":float(df["skills"].apply(len).mean()),
        "seniority_distribution":df["seniority"].value_counts().to_dict()
    }

# for saving the validated job postings to CSV and JSONL files for further use
def save_outputs(rows:List[Dict], stem:str="synthetic_jobs"):
    pd.DataFrame(rows).to_csv(f"{stem}.csv", index=False)
    with open(f"{stem}.jsonl", "w", encoding="utf-8") as f:
        for r in rows:
            # json.dumps: converts a Python object into a JSON string
            f.write(json.dumps(r, ensure_ascii=True) + "\n")

In [ ]:
# for orchestrating the entire process: creating the client, generating data, validating it, reporting quality, and saving outputs
def run():
    client = make_client()
    model = "openai/gpt-oss-20b"
    raw = generate_batch(
        client,
        model,
        n=25,
        domain="software engineering",
        seniority_mix="20% junior, 50% mid, 30% senior",
        temperature=0.2,
    )
    clean = validate_row(raw)
    report = quality_report(clean)
    save_outputs(clean, stem="jobs_software_engineering")
    print("Report:", report)
    print(f"Saved {len(clean)} valid job postings.")

# entry point of the script
if __name__ == "__main__":
    run()

In [ ]:
import gradio as gr

# for generating data and returning both the quality report and a preview of the valid job postings for display in the UI
def generate_ui(model, n_rows, domain, seniority_mix, temperature):
    try:
        client = make_client()
        raw = generate_batch(client, model, int(n_rows), domain, seniority_mix, temperature=float(temperature))
        clean = validate_row(raw)
        report = quality_report(clean)
        save_outputs(clean, stem="synthetic_jobs")
        preview = clean[:10]
        return json.dumps(report, indent=2), preview
    except Exception as e:
        return json.dumps({"error": str(e)}, indent=2), []

# building the Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("# Synthetic Job Posting Generator")
    model = gr.Textbox(label="Model", value="openai/gpt-oss-20b")
    n_rows = gr.Slider(label="Number of rows", minimum=10, maximum=500, step=10, value=50)
    domain = gr.Textbox(label="Domain", value="software engineering")
    seniority_mix = gr.Textbox(label="Seniority mix", value="20% junior, 50% mid, 30% senior")
    temperature = gr.Slider(label="Temperature", minimum=0.1, maximum=1.0, step=0.1, value=0.2)
    generate_button = gr.Button("Generate")
    report = gr.Code(label="Quality Report")
    preview = gr.JSON(label="Preview of valid job postings")
    generate_button.click(generate_ui, inputs=[model, n_rows, domain, seniority_mix, temperature], outputs=[report, preview])

demo.launch()